# Comparing survival curves 
## Introduction


## Prednisolone case study


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data_altman = pd.DataFrame({
    'durations': [ # Time in months
        # control:
        2,3,4,7,10,22,28,29,32,37,40,41,54,61,63,71,127,140,146,158,167,182,
        # prednisolone:
        2,6,12,54,56,68,89,96,96,125,128,131,140,141,143,145,146,148,162,
        168,173,181,
    ],
    'event_observed': [
        1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0, # 1 = death
        1,1,1,1,0,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,0,0, # 0 = censored
    ],
    # Dummy encode treatment group as True or False
    'prednisolone': [False]*22 + [True]*22})

print("First rows of the Altman dataframe:")
print(data_altman.head())

### Handling dates in survival analysis


### Kaplan-Meier analysis


In [ ]:

from lifelines import KaplanMeierFitter

# Note that we have two separate Fitter object each with its own label
kmf_ctrl = KaplanMeierFitter(label='placebo (KM)')
kmf_pred = KaplanMeierFitter(label='prednisolone (KM)')

# Define censor styles once for reuse
censor_styles = {'marker': 2, 'ms': 6}

# Filter data by creating a boolean mask
ix = data_altman['prednisolone']

# Fit for prednisolone and control groups
kmf_pred.fit(
    durations=data_altman.loc[ix, 'durations'],
    event_observed=data_altman.loc[ix, 'event_observed'])

kmf_ctrl.fit(
    data_altman[~ix]['durations'],
    data_altman[~ix]['event_observed'])

# Plot both KM curves on the same figure
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 3))

kmf_ctrl.plot_survival_function(
    ax=ax,
    show_censors=True,
    lw=2, ci_alpha=.1, censor_styles=censor_styles)

kmf_pred.plot_survival_function(
    ax=ax,
    show_censors=True,
    lw=2, ci_alpha=.1, censor_styles=censor_styles)

# Customize plot appearance
plt.xlabel('Months of follow-up',)
plt.ylabel('Proportion survival')
plt.title('Kaplan-Meier analysis');

# plt.savefig('survival.svg');
# It's also possible to generate a Plotly plot using the 
# .mpl_to_plotly() method, 
# see https://plot.ly/ipython-notebooks/survival-analysis-r-vs-python/

### Beyond the curves
#### Median survival


In [ ]:
# Extract median survival times
median_survival_ctrl = kmf_ctrl.median_survival_time_
median_survival_pred = kmf_pred.median_survival_time_

print("Median survival time (in months):")
print(f"  Group {kmf_ctrl._label} = {median_survival_ctrl}")
print(f"  Group {kmf_pred._label} = {median_survival_pred}")

#### Survival predictions


In [ ]:
# Simply converting 5 years into months
month = 5 * 12

print(f"Survival prediction at month {month}:")
print(f"  Group {kmf_ctrl._label} = {100*kmf_ctrl.predict(month):.1f}%")
print(f"  Group {kmf_pred._label} = {100*kmf_pred.predict(month):.1f}%")

#### Survival function


In [ ]:
print(f"First rows of the survival function of {kmf_pred._label}:")
print(kmf_pred.survival_function_.head())

## Non-parametric log-rank test
### A little bit of theory
### Log-rank test with lifelines


In [ ]:
from lifelines.statistics import logrank_test

# Perform the log-rank test
results = logrank_test(
    durations_A=data_altman.loc[ix, 'durations'],
    durations_B=data_altman.loc[~ix, 'durations'],
    event_observed_A=data_altman.loc[ix, 'event_observed'],
    event_observed_B=data_altman.loc[~ix, 'event_observed'])

# Print the summary of the results
print(f"Log-rank test comparing {kmf_pred._label} with {kmf_ctrl._label}")
print(results.summary)

# results.print_summary()
# results.p_value
# results.test_statistic

### Mantel-Haenszel method
### Gehan-Breslow-Wilcoxon method


In [ ]:
# Using the Gehan-Breslow-Wilcoxon method
results_wilcoxon = logrank_test(
    durations_A=data_altman.loc[ix, 'durations'],
    durations_B=data_altman.loc[~ix, 'durations'],
    event_observed_A=data_altman.loc[ix, 'event_observed'],
    event_observed_B=data_altman.loc[~ix, 'event_observed'],
    weightings='wilcoxon')

print(results_wilcoxon.summary)

## Hazard ratio for quantifying relative risk


### Nelson-Aalen estimator
#### The mathematics behind the model
#### Using the `NelsonAalenFitter` in lifelines


In [ ]:

from lifelines import NelsonAalenFitter

naf_ctrl = NelsonAalenFitter(label='placebo (NAF)')
naf_pred = NelsonAalenFitter(label='prednisolone (NAF)')

# Fit Nelson-Aalen estimators for each group
naf_pred.fit(
    data_altman[ix]['durations'],
    event_observed=data_altman[ix]['event_observed'])
naf_ctrl.fit(
    data_altman[~ix]['durations'],
    event_observed=data_altman[~ix]['event_observed'])

# Plot cumulative hazard functions on the same figure
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 3))
naf_ctrl.plot(ax=ax, ci_alpha=.1)
naf_pred.plot(ax=ax, ci_alpha=.1)

plt.xlabel('Time (months)')
plt.ylabel('Cumulative hazard')
plt.title('Nelson-Aalen cumulative hazard estimates');

In [ ]:
# Print cumulative hazard data (first 5 rows) for prednisolone group
print(f"Cumulative hazard data (first rows):")
print(naf_pred.cumulative_hazard_.head())

#### Connection to Kaplan-Meier survival


#### Link with hazard ratio


### Cox proportional hazards model
#### The maths behind Cox PH


#### Interpreting the results of the Cox PH model


In [ ]:
from lifelines import CoxPHFitter

# Fit the CoxPHFitter
cph = CoxPHFitter()

cph.fit(
    df=data_altman,
    duration_col='durations',
    event_col='event_observed',
    # formula='prednisolone'
)

# Print the summary to see the coefficient and its significance
cph.print_summary(decimals=3, style='ascii')

#### Checking proportional hazards assumption


In [ ]:
cph.check_assumptions(
    training_df=data_altman,
    p_value_threshold=.05,  # !default 0.01
    show_plots=False);

#### Visualizing the partial effect of covariate


In [ ]:

# Visualize the partial effect of the predictor on survival
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

cph.plot_partial_effects_on_outcome(
    covariates=['prednisolone'],
    values=[False, True],
    plot_baseline=True,
    ax=ax)

plt.title("Impact of 'prednisolone'")
plt.xlabel('Time (months)')
plt.ylabel('Survival probability');

#### Baseline


## The Weibull parametric model
### Weibull distribution


In [ ]:
#| fig-subcap: 
#|   - "Varying scale"
#|   - "Varying shape"
#| layout-ncol: 2

import numpy as np
import seaborn as sns
from scipy.stats import weibull_min

# Create a range of values for x
x = np.arange(0, 5, 0.01)

# Use a context manager to change color palette only in this figure
with sns.color_palette("magma_r"):


    # Left subplot: varying lambda with rho = 2
    plt.figure(figsize=(3.5, 3))
    for scale in [1, 2, 3, 4]:
        # Calculate Weibull PDF
        weibull_pdf = weibull_min(scale=scale, c=2).pdf(x)
        plt.plot(x, weibull_pdf, label=f'λ={scale}')

    plt.title('Weibull PDF (ρ =2)')
    plt.xlabel('x')
    plt.yticks([])
    plt.ylabel('Probability')
    plt.margins(x=0.02, y=0)
    plt.legend()
    plt.show()

    # ---
    # Right subplot: varying rho with lambda = 1
    plt.figure(figsize=(3.5, 3))
    for rho in [0.5, 1, 2, 4]:
        weibull_pdf = weibull_min.pdf(x, scale=1, c=rho)
        plt.plot(x, weibull_pdf, label=f'ρ={rho}')

    plt.title('Weibull PDF (λ=1)')
    plt.xlabel('x')
    plt.yticks([])
    plt.ylabel('Probability')
    plt.margins(x=0.02, y=0)
    plt.legend()
    plt.show()

### Connection to hazard and cumulative hazard functions


### Application to the prednisolone data


In [ ]:

from lifelines import WeibullFitter

# Fit Weibull models for each group
wb_ctrl = WeibullFitter()
wb_pred = WeibullFitter()

wb_ctrl.fit(
    durations=data_altman[~ix]['durations'],
    event_observed=data_altman[~ix]['event_observed'],
    label='placebo (Weibull)')

wb_pred.fit(
    data_altman[ix]['durations'],
    data_altman[ix]['event_observed'],
    label='prednisolone (Weibull)')

# Plot survival functions
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 3))
wb_ctrl.plot_survival_function(ax=ax)
wb_pred.plot_survival_function(ax=ax)

# Overlay KM curves on the same plot
kmf_ctrl.plot(ax=ax, ci_show=False, color='steelblue', ls='--', alpha=.75)
kmf_pred.plot(ax=ax, ci_show=False, color='darkorange', ls='--', alpha=.75)

plt.title("Weibull Fit vs. Kaplan-Meier")
plt.xlabel("Time (months)")
plt.ylim((0, 1))
plt.ylabel("Fraction survival");

### Goodness-of-fit


In [ ]:

from lifelines.plotting import qq_plot

fig, axes = plt.subplots(1, 2, figsize=(6, 3), sharey=False)

# Q-Q plot for placebo group
qq_plot(wb_ctrl, ax=axes[0])
axes[0].set_title(f"Q-Q - {wb_ctrl.label}")

# Q-Q plot for prednisolone Group
qq_plot(wb_pred, ax=axes[1])
axes[1].set_title(f"Q-Q - {wb_pred.label}")

# Adjust layout and display the plot
plt.tight_layout();

### Parameters estimates 


In [ ]:
wb_ctrl.print_summary(style='ascii')

In [ ]:
wb_pred.print_summary(style='ascii')

### Weibull regression with covariates
#### Accelerated failure time model
#### Weibull AFT with one covariate


In [ ]:
from lifelines import WeibullAFTFitter

# Fit the Weibull AFT model
aft = WeibullAFTFitter()
aft.fit(
    df=data_altman,
    duration_col='durations',
    event_col='event_observed')

aft.print_summary(style='ascii', decimals=2)

In [ ]:
# Explicit
aft.fit(
    df=data_altman,
    duration_col='durations',
    event_col='event_observed',
    formula='prednisolone')

aft.print_summary(style='ascii', decimals=2)

#### Weibull AFT with multiple covariates
##### Rossi dataset


In [ ]:
from lifelines.datasets import load_rossi

data_rossi = load_rossi()

print("First rows of the Rossi dataset:")
print(data_rossi.head())

##### Fitting the Weibull AFT model to Rossi data


In [ ]:
aft_rossi = WeibullAFTFitter()
aft_rossi.fit(
    df=data_rossi,
    duration_col='week',
    event_col='arrest',
    # formula="fin + wexp + age + prio + age:prio",  # Same as below
    formula="fin + wexp + age * prio")

aft_rossi.print_summary(style='ascii', decimals=2)

##### Visualizing the influence of covariates


In [ ]:

_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))
aft_rossi.plot(ax=ax)
plt.title("Weibull AFT forest plot - Rossi");

In [ ]:

_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))
aft_rossi.plot_partial_effects_on_outcome(
    covariates='prio',
    values=range(0, 16, 3),
    cmap='viridis',
    ax=ax)

plt.title("'prio' partial effects on survival")
plt.xlabel('Time (weeks)')
plt.ylabel('Survival probability');

##### Visualizing the interaction between covariates


In [ ]:

# Create a list of tuples representing combinations of 'prio' / 'age' values
values_grid = [(p, a) for p in range(0, 16, 3) for a in (20, 40)]

# Plot the partial effects on outcome for the combined covariates
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))
aft_rossi.plot_partial_effects_on_outcome(
    covariates=['prio', 'age'],
    values=values_grid,
    cmap='Paired_r',
    ax=ax)

# Enhance the plot with labels and title
plt.title("'prio' and 'age' interaction")
plt.xlabel('Time (weeks)')
plt.ylabel('Survival probability')

plt.legend([f"prio={p}, age={a}" for p, a in values_grid], fontsize=6);

##### Applying the Weibull AFT model for prediction


In [ ]:
# Define the profiles of three new individuals
new_individuals = pd.DataFrame(
    data={
        'fin': [.5, .5, .5],
        'age': [20, 30, 40],
        'race':[1, 1, 1],
        'wexp':[0, 0, 0],
        'mar': [0, 0, 0],
        'paro':[1, 1, 1],
        'prio':[10, 10, 10]},
    index=[301, 302, 303])

# Display the DataFrame of new individuals
print("DataFrame of new individuals for prediction:")
print(new_individuals)
print()  # Add an empty line for readability

# Predict median survival time for the new individuals
# Note: predict_median expects a DataFrame, not a Series
median_survival = aft_rossi.predict_median(new_individuals)

# Display the predicted median survival times
print("Predicted median survival times:")
print(median_survival.round(1))

In [ ]:

# Predict survival function for a new combination of covariates
survival_function = aft_rossi.predict_survival_function(new_individuals)

# Plot the predicted survival function
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))
survival_function.plot(ax=ax, cmap='viridis')
plt.title(f"Predicted survival functions")
plt.xlabel('Time (weeks)')
plt.ylabel('Survival probability');

## Alternative parametric survival models


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from lifelines import (
    ExponentialFitter,
    LogNormalFitter,
    LogLogisticFitter,
    GeneralizedGammaFitter)

# Instantiate each fitter
wb = WeibullFitter()
exp = ExponentialFitter()
lognorm = LogNormalFitter()
loglogistic = LogLogisticFitter()
gamma = GeneralizedGammaFitter()

# Fit to data and display the AIC
print("AIC values for different models fitting 'prednisolone' data:\n")
for model in [wb, exp, lognorm, loglogistic, gamma]:
    model.fit(
        durations=data_altman[ix]['durations'],
        event_observed=data_altman[ix]['event_observed'])
    print(model.__class__.__name__, '\t', model.AIC_.round(2))

In [ ]:

# Plot for prednisolone group
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

kmf_pred.plot(
    style='--', color='darkgrey',
    ci_show=False,
    label='Kaplan-Meier',  # We can change the label of the model
    ax=ax)

for model, color in zip(
    [wb, exp, lognorm],
    ['yellowgreen', 'firebrick', 'slategray']):
    model.plot_survival_function(
        ax=ax,
        ci_show=False,
        color=color,
        label=model.__class__.__name__.removesuffix('Fitter'))

plt.title("Prednisolone group")
plt.xlabel('Time (months)')
plt.ylabel('Survival probability')
plt.legend(fontsize=9);

In [ ]:
from lifelines.utils import find_best_parametric_model

# Find the best parametric model for the placebo group
best_model_ctrl, best_aic_ctrl = find_best_parametric_model(
    event_times=data_altman[~ix]['durations'],
    event_observed=data_altman[~ix]['event_observed'],
    scoring_method="AIC")

# Print the best model and its AIC for the placebo group
print("Best Parametric model for placebo group:")
print(best_model_ctrl)
print(f"AIC: {best_aic_ctrl:.2f}")

# Find the best parametric model for the 'prednisolone' group
best_model_pred, best_aic_pred = find_best_parametric_model(
    event_times=data_altman[ix]['durations'],
    event_observed=data_altman[ix]['event_observed'],
    scoring_method="AIC")

# Print the best model and its AIC for the 'prednisolone' group
print("\nBest Parametric model for prednisolone group:")
print(best_model_pred)
print(f"AIC: {best_aic_pred:.2f}")

## Conclusion
